## Short-Term Rental Investment Analysis in San Luis Obispo County

I live on the Central Coast of California, in San Luis Obispo County, and wanted to assess whether short-term rentals (STRs) are a good investment in the area.

### **Step 1: Research Local Regulations**
Since STR regulations vary by city, understanding local restrictions and taxes is crucial. For example:
- **San Luis Obispo (SLO) City** prohibits non-owner-occupied STRs.
- **Unincorporated areas** like Cambria and Cayucos impose distance requirements between STRs.

### **Step 2: Estimating Return on Investment (ROI)**
For cities with relatively fewer restrictions, I analyzed potential ROI by developing a code that estimates common expenses, including:
- **Fixed costs:** Mortgage, closing fees, insurance, and property taxes.
- **STR-specific taxes & fees:** Transient Occupancy Tax (TOT) and Tourism Marketing District (TMD) fees.


In [1]:
import pandas as pd



def calculate_loan_payment(loan_amount, monthly_interest_rate, loan_term_months):
    """Calculate monthly mortgage payment based on loan amount, interest rate, and term."""
    return (loan_amount * monthly_interest_rate) / (1 - (1 + monthly_interest_rate) ** (-loan_term_months))

def calculate_monthly_expenses(home_price, property_tax_rate, insurance_cost, rental_income, management_fee_percent, maintenance_cost_percent, TMD_rate, TOT_rate):
    """Calculate total monthly expenses including taxes, insurance, management fees, maintenance, TMD, and TOT."""
    property_tax_monthly = (home_price * property_tax_rate) / 12
    insurance_monthly = insurance_cost / 12
    management_fees = rental_income * management_fee_percent
    maintenance_costs = rental_income * maintenance_cost_percent
    TMD_monthly = rental_income * TMD_rate
    TOT_monthly = rental_income * TOT_rate  # New: Transient Occupancy Tax (TOT)
    return property_tax_monthly + insurance_monthly + management_fees + maintenance_costs + TMD_monthly + TOT_monthly

def calculate_roi(net_income, investment):
    """Calculate ROI as a percentage."""
    return (net_income / investment) * 100

In [2]:
# Input parameters (adjust if needed)
occupancy_rate = 0.6
occupied_nights_per_month = occupancy_rate * 30  # Assume 60% occupancy over 30 days
closing_costs_percent = 0.03
management_fee_percent = 0.20  # Assume 20% for management
maintenance_cost_percent = 0.10  # Assume 10% for maintenance
TMD_rate = 0.015  # Assume 1.5% TMD rate in SLO unincorporated areas
TOT_rate = 0.09  # Assume 9% Transient Occupancy Tax in SLO

# Property details for each city
cities = ["San Luis Obispo", "Paso Robles", "Arroyo Grande", "Atascadero", "Grover Beach"]
home_prices = [1089500, 745606, 1010559, 757219, 757913]
down_payment_percent = 0.20
interest_rate = 0.065
loan_term_years = 30
property_tax_rate = 0.011
insurance_costs = [1800, 1600, 1750, 1600, 1600]
nightly_rates = [260, 225, 240, 220, 230]


In [3]:
# Calculations
loan_amounts = [price * (1 - down_payment_percent) for price in home_prices]
rental_income_monthly = [rate * occupied_nights_per_month for rate in nightly_rates]
monthly_interest_rate = interest_rate / 12
loan_payments = [calculate_loan_payment(loan, monthly_interest_rate, loan_term_years * 12) for loan in loan_amounts]

monthly_expenses = [
    calculate_monthly_expenses(price, property_tax_rate, insurance, income, management_fee_percent, maintenance_cost_percent, TMD_rate, TOT_rate)
    for price, insurance, income in zip(home_prices, insurance_costs, rental_income_monthly)
]

net_income_monthly = [income - expense for income, expense in zip(rental_income_monthly, monthly_expenses)]
annual_net_income = [net * 12 for net in net_income_monthly]

initial_ROI_percent = [calculate_roi(net_income, price) for net_income, price in zip(annual_net_income, home_prices)]

initial_investment = [(price * down_payment_percent) + (price * closing_costs_percent) for price in home_prices]

# ROI: income before mortgage
final_ROI_percent = [calculate_roi(net_income, invest) for net_income, invest in zip(annual_net_income, initial_investment)]

# Cash-on-Cash Return Calculation: income after portgage
cash_flow_monthly = [net - mortgage for net, mortgage in zip(net_income_monthly, loan_payments)]
cash_flow_annual = [cf * 12 for cf in cash_flow_monthly]
cash_on_cash_return = [calculate_roi(cash_flow, invest) for cash_flow, invest in zip(cash_flow_annual, initial_investment)]

management_fees = [income * management_fee_percent for income in rental_income_monthly]
maintenance_costs = [income * maintenance_cost_percent for income in rental_income_monthly]
TMD_monthly_expenses = [income * TMD_rate for income in rental_income_monthly]
TOT_monthly_expenses = [income * TOT_rate for income in rental_income_monthly]

# DataFrame creation
df = pd.DataFrame({
    "City": cities,
    "Home Price ($)": home_prices,
    "Loan Amount ($)": loan_amounts,
    "Monthly Rental Income ($)": rental_income_monthly,
    "Monthly Mortgage Payment ($)": loan_payments,
    "Monthly Property Tax ($)": [(price * property_tax_rate) / 12 for price in home_prices],
    "Monthly Insurance ($)": [cost / 12 for cost in insurance_costs],
    "Property Management ($)": management_fees,
    "Monthly Maintenance ($)": maintenance_costs,
    "Monthly TMD Expense ($)": TMD_monthly_expenses,
    "Monthly TOT Expense ($)": TOT_monthly_expenses,  # New: TOT included
    "Total Monthly Expenses ($)": monthly_expenses,
    "Net Monthly Income ($)": net_income_monthly,
    "Annual Net Income ($)": annual_net_income,
    "Final ROI (%)": final_ROI_percent,
    "Cash Flow Monthly ($)": cash_flow_monthly,  # New: Cash flow added
    "Cash-on-Cash Return (%)": cash_on_cash_return  # New: CoC Return
})

print(df[["City", "Final ROI (%)", "Cash-on-Cash Return (%)"]])

              City  Final ROI (%)  Cash-on-Cash Return (%)
0  San Luis Obispo       7.833949               -18.548021
1      Paso Robles      11.146658               -15.235312
2    Arroyo Grande       7.735130               -18.646840
3       Atascadero      10.533390               -15.848580
4     Grover Beach      11.256630               -15.125339


The calculation is based on publicly available data and reasonable assumptions. It uses a two-bedroom property as an example, with its median price derived from my other code (with slight modifications): [Zillow Property Listing Analysis](https://github.com/yi-hui-wang/Real-Estate-Analysis/blob/main/Notebooks/ZillowPropertyListing_Statistics_SegmentationAnalysis.ipynb).

This estimate does not account for seasonal fluctuations in occupancy rates or rental pricing.


### **Findings & Conclusion**
The estimate indicates that if purchasing an STR with a mortgage, the annual cash flow return after mortgage payments is lower than -15%. Given the high property prices, interest rates, operating costs, and taxes, investing in STRs in SLO County appears to be a poor financial choice.